# K-Means and Hierarchical Clustering with the World Cup Dataset
1. Import the Excel dataset
2. Store it in a pandas dataframe
3. Check and handle missing values
4. Select feature columns
5. Standardize the feature values
6. Use the Elbow Method to choose K
7. Perform K-Means clustering
8. Perform hierarchical clustering and create a dendrogram
9. Compare both clustering results

In [ ]:
# IMPORT PACKAGES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# sklearn packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

# hierarchical clustering packages
from scipy.cluster.hierarchy import linkage, dendrogram

## 1. Import the cleaned Excel dataset

In [ ]:
# import the cleaned Excel dataset
worldcup = pd.read_excel("kmeans_dataset_cleaned.xlsx")

worldcup

## 2. Store the data in a pandas dataframe

In [ ]:
# Check the column names, and data types of the dataset
print("Column names:")
print(worldcup.columns.tolist())

print()
print("Data types:")
print(worldcup.dtypes)

## 3. Check for missing values and replace missing values with average values

In [ ]:
# CHECK MISSING VALUES BEFORE CLEANING
print("Missing values before cleaning:")
worldcup.isna().sum()

In [ ]:
# check how many feature values are missing PER STUDENT (per row)
worldcup_feature_columns = ["germany_chance_win", "football_interest", "dallas_home_distance",
                    "team_support_strength", "group_watching", "wc_excitement",
                    "culture_interest", "sustainability_importance"]

missing_per_row = worldcup[worldcup_feature_columns].isnull().sum(axis=1)
pd.DataFrame({"Student": worldcup["Student"], "missing_values": missing_per_row})

In [ ]:
# one value in group_watching was saved like text, so I take the first number from it
worldcup["group_watching"] = worldcup["group_watching"].astype(str).str.extract(r"(\d+\.?\d*)")

# convert all feature columns to numeric values
for col in worldcup_feature_columns:
    worldcup[col] = pd.to_numeric(worldcup[col], errors="coerce")

# check missing values again
worldcup[worldcup_feature_columns].isna().sum()

In [ ]:
# replace missing values with the average of each column
worldcup[worldcup_feature_columns] = worldcup[worldcup_feature_columns].fillna(worldcup[worldcup_feature_columns].mean())

# check if missing values are gone
worldcup[worldcup_feature_columns].isna().sum()

## 4. Identify the feature columns for the analysis

In [ ]:
# Select the feature columns to be used for clustering.
# K-Means clustering can only work with numerical values, so we do not include the student name.
X = worldcup[worldcup_feature_columns]

print("Feature columns used for clustering:")
print(worldcup_feature_columns)

X

## 5. Standardize the feature values

In [ ]:
# Use the standard scaler to normalize the feature values.
worldcup_scaler = StandardScaler()

# Call the standard scaler
X_worldcup_scaled = worldcup_scaler.fit_transform(X)

# Put the scaled data into a dataframe for better readability.
X_worldcup_scaled_df = pd.DataFrame(
    X_worldcup_scaled,
    columns=worldcup_feature_columns,
    index=worldcup["Student"]
)

X_worldcup_scaled_df.head()

## 6. Use the Elbow Method to identify a suitable number of clusters K

In [ ]:
# list for inertia values
inertias = []

# K values that I want to test
k_values = range(1, 10)

# run K-Means for every K
for k in k_values:
    kmeans = KMeans(n_clusters = k, random_state = 56, n_init = 10)
    kmeans.fit(X_worldcup_scaled_df)
    inertias.append(kmeans.inertia_)

inertias

In [ ]:
# PLOT
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for World Cup Dataset")
plt.xticks(list(k_values))
plt.show()

## 7. Perform K-Means clustering

In [ ]:
# choose K
k = 3

# create the K-Means model
worldcup_kmeans = KMeans(n_clusters = k, random_state = 40, n_init = 10)

# add the cluster result to the dataframe
worldcup["kmeans_cluster"] = worldcup_kmeans.fit_predict(X_worldcup_scaled)

worldcup[["Student", "kmeans_cluster"] + worldcup_feature_columns]

In [ ]:
# mean values of each cluster
kmeans_summary = worldcup.groupby("kmeans_cluster")[worldcup_feature_columns].mean().round(2)

kmeans_summary

# show which students are in each cluster
for cluster in sorted(worldcup["kmeans_cluster"].unique()):
    names = worldcup.loc[worldcup["kmeans_cluster"] == cluster, "Student"].tolist()
    print("Cluster", cluster, ":", names)

In [ ]:
# reduce the scaled data to 2 dimensions for the plot
pca = PCA(n_components = 2, random_state = 40)
X_pca = pca.fit_transform(X_worldcup_scaled)

worldcup["pca_1"] = X_pca[:, 0]
worldcup["pca_2"] = X_pca[:, 1]

plt.figure(figsize = (9, 6))
plt.scatter(
    worldcup["pca_1"],
    worldcup["pca_2"],
    c = worldcup["kmeans_cluster"],
    s = 80
)

# add student names to the points
for i, student in enumerate(worldcup["Student"]):
    plt.annotate(student, (worldcup["pca_1"][i] + 0.05, worldcup["pca_2"][i] + 0.05), fontsize = 8)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means clusters - World Cup dataset")
plt.show()

## 8. Perform hierarchical clustering and create a dendrogram

In [ ]:
worldcup_linked = linkage(
    X_worldcup_scaled,
    method="ward"
)

In [ ]:
# plot dendrogram
plt.figure(figsize = (12, 7))

dendrogram(
    worldcup_linked,
    labels = worldcup["Student"].values,
    leaf_rotation = 90
)

plt.title("Hierarchical clustering dendrogram")
plt.xlabel("Student")
plt.ylabel("Euclidean distance")
plt.show()

## 9. Compare the results of K-Means and hierarchical clustering

In [ ]:
# Perform Agglomerative Clustering to get hierarchical cluster labels
hierarchical_model = AgglomerativeClustering(n_clusters=3)
worldcup["hierarchical_cluster"] = hierarchical_model.fit_predict(X_worldcup_scaled)

cluster_comparison = pd.crosstab(
    worldcup["kmeans_cluster"],
    worldcup["hierarchical_cluster"],
    rownames=["K-Means cluster"],
    colnames=["Hierarchical cluster"]
)

print("Comparison between K-Means and Hierarchical Clustering:")
display(cluster_comparison)

ari_score = adjusted_rand_score(
    worldcup["kmeans_cluster"],
    worldcup["hierarchical_cluster"]
)

print("Adjusted Rand Index:", round(ari_score, 3))

In [ ]:
# final dataframe with both clustering results
worldcup_final = worldcup[["Student", "kmeans_cluster", "hierarchical_cluster"] + worldcup_feature_columns]

worldcup_final

## 1. Which K-value did you choose, and why?
I chose K = 3. In the Elbow Method, the inertia decreases as K increases, but after around 3 clusters the improvement becomes smaller. K = 3 is also easier to interpret than a larger number of clusters. It separates the dataset into three understandable groups.

## 2. What are the main characteristics of the clusters? Can you interpret them?
The K-Means result gives three main groups:

- Cluster 0: General World Cup interest group  
  This is the largest group. Most students are in this cluster. They usually have medium to high football interest, strong group-watching preference, and medium to high culture and sustainability interest.

- Cluster 1: Germany-belief / high-interest group  
  This small group contains students who think Germany has a chance to win. They also show high football interest and high group-watching preference, but their World Cup excitement is not extremely high.

- Cluster 2: Low-engagement group / outlier  
  This group contains the student with very low values in many preference variables, especially football interest, team support, group watching, excitement, culture interest, and sustainability importance. This makes the case very different from the rest of the dataset.

## Do K-Means and hierarchical clustering lead to similar groups?
Yes, the two methods lead to very similar groups. When the same number of clusters is used for hierarchical clustering, the crosstab shows that the observations are grouped in the same way. The Adjusted Rand Index is also very high, which means both methods found almost identical cluster structures in this dataset.